In [ ]:
import sys
from pathlib import Path

src = Path.cwd().parents[2] / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

import pandas as pd
import myflopy as mf
from canonical_notebook_style import notebook_header

notebook_header(
    'Demo',
    'How sure are we?',
    'Calibrating a groundwater model with an ensemble — and getting an honest '
    'error bar on the prediction we actually care about.'
)

## What this is

A groundwater model is a computer model of water moving underground. We build one
to answer a question — *how far will this contaminant travel?*, *how much can we
pump before the creek dries up?*

The awkward part is that the ground is opaque. The properties that drive the
answer are buried, and we only ever measure a handful of them. So we do the next
best thing: we take the measurements we **do** have (water levels in wells) and
adjust the unknown properties until the model reproduces them. That is
**calibration**.

And then comes the question this notebook is about:

> When the model finally matches my measurements — **how much should I trust its
> prediction?**

The honest answer is never a single number. Many different underground
configurations reproduce the same handful of well measurements. A single
"calibrated" model quietly hides all the other answers that fit just as well.

So instead of calibrating one model, we calibrate **hundreds at once** and look
at the spread. Everything below is that idea.

---

## 0. Housekeeping

Nothing here runs a model. The ensemble was run ahead of time; this notebook
reopens it from disk. That is a normal way to work — the run lives beside the
model it calibrated, and `model.pest_runs` finds it again later.

In [ ]:
ARTIFACTS = Path('../artifacts/demo_ies')

# Reopen the model exactly as it was left on disk.
model_dir = next(p for p in (ARTIFACTS / 'model').iterdir()
                 if p.is_dir() and (p / 'mfsim.nam').exists())
model = mf.load_mf6_run(model_dir, verbosity_level=0)

# Every calibration ever run against this model, discovered from disk.
runs = model.pest_runs
for run in runs:
    print(run)

In [ ]:
# Reopen one of them for review. `ies` below is identical to what you would have
# had in memory right after the run finished.
ies = next(r for r in runs if r.name == 'demo_ies').review(model=model)
print(ies.settings)

---

## 1. The model

A river valley: an alluvial aquifer between two hillsides, with a stream running
down it, a lake, some pumping wells, and drains. Four layers. The grid is
**unstructured** — Voronoi polygons rather than rectangles — so it can follow the
stream and tighten up where we need detail, instead of forcing the whole domain
onto one rectangular mesh.

This is the shape of a model you would actually build for a real site.

In [ ]:
# Read the MODFLOW packages straight off the model's name file. Reopening a run
# loads only what the readers need, so ask the file rather than the loaded object.
name_file = model_dir / f'{model_dir.name}.nam'
packages = [line.split()[0] for line in name_file.read_text().splitlines()
            if line.strip().upper().endswith(('6',)) or ' 6 ' in line]
packages = [p for p in packages if p.upper().endswith('6')]

pd.Series({
    'grid cells per layer': model.vor.ncpl,
    'layers': model.gwf.modelgrid.nlay,
    'stress periods': model.nper,
    'MODFLOW packages': len(packages),
}, name='the model')

In [ ]:
# Not a toy: a full surface-water/groundwater model.
print(' '.join(p.replace('6', '') for p in packages))

In [ ]:
# The grid itself. Cells are smaller along the stream corridor, where we need
# the detail, and coarser out at the valley walls where we do not.
model.vor.plot()

In [ ]:
# Simulated water table at the end of the run. Water enters up-valley and along
# the hillsides, and leaves through the stream, the lake and the wells.
model.hds.map(per=model.nper - 1, layer=0)

---

## 2. What we don't know

The property that matters most here is **hydraulic conductivity** (K) — how
easily water moves through the ground. Gravel is high, clay is low. It varies
from place to place across the valley, and we cannot see it.

What we *do* have is a set of wells with measured water levels. The job of
calibration is to work backwards: find the K field that explains those levels.

The catch, and the whole reason for this notebook: **that problem has many
answers.** Different K fields produce nearly identical water levels at our
handful of wells while disagreeing completely somewhere we didn't measure.

In [ ]:
# The observations we are calibrating against: water levels at monitoring wells.
residuals = ies.obs_residuals()
pd.Series({
    'monitoring locations': residuals['location'].nunique(),
    'individual measurements': int(ies.pst.nnz_obs),
    'adjustable parameters': int(ies.pst.npar_adj),
}, name='the calibration problem')

Note the shape of that table: **more unknowns than measurements.** That is the
normal situation in groundwater modelling, and it is exactly why a single
"best-fit" answer is misleading.

---

## 3. One model, or many?

Instead of one model we build an **ensemble** — here, 30 complete copies of the
model, each with a different, plausible K field. Each copy is called a
**realization**.

The spread of the ensemble *before* we look at any data is the **prior**: it
encodes what we believed about K beforehand — roughly how large, and how smoothly
it varies in space.

Below, each grey line is one realization's simulated water level through time;
the red markers are the actual measurements. This is the cheapest and most
valuable check in the whole workflow: **does the prior even contain the data?**
If every grey line sits above the red dots, no amount of calibration will save
us — the model or the assumed ranges are wrong, and we should find that out now
rather than three weeks later.

In [ ]:
ies.plot_prior_vs_obs()

---

## 4. Nudging every realization toward the data

**PESTPP-IES** (Iterative Ensemble Smoother) takes the whole ensemble and moves
every realization toward the measurements at once, over a few iterations. Each
realization is fitted to a slightly different, noise-perturbed copy of the data,
so uncertainty *in the measurements* flows through into uncertainty in the answer.

In myflopy that is one line — `cal.run_ies(reals=30, iterations=3, workers=12)` —
and the realizations run in parallel.

To see whether it worked we need one number per realization for "how badly does
this model miss the measurements". That number is **phi** (Φ): the sum of squared
mismatches. Lower is better.

In [ ]:
# Phi through the iterations. Each line is one realization.
ies.plot_phi()

### The first distribution: misfit, before and after

The histogram below is the same information, as a distribution. Grey is the
prior — the ensemble before history matching. Blue is the posterior — after.

**What to look for:** a clear shift left means the ensemble genuinely learned
from the data. But a posterior that collapses into a narrow spike at very low
phi is a *warning*, not a triumph: with an imperfect model you should not be able
to drive the misfit to zero, and a model that fits the calibration data perfectly
usually predicts badly. Some residual misfit is honest.

In [ ]:
ies.plot_phi_distribution()

---

## 5. The payoff: a prediction with an error bar

Here is the point of all of it.

A **forecast** is something we want to predict but have no measurement for — a
water level somewhere we never drilled, a future flow. We declared one before
calibrating, and because every realization is a complete working model, every
realization produces its own value for it. That gives us a *distribution* rather
than a number.

Grey is what we believed before calibration. Blue is what we believe after.
**The narrowing is the value of the data.**

In [ ]:
name = ies.forecast_names[0]
ies.forecast(name).plot()

In [ ]:
# Every forecast, prior versus posterior, as numbers.
ies.forecasts()

Read that as: *"before calibrating we could only say the level was somewhere in
this wide range; after using the well data we can say it is in this narrower
one."* That sentence — with an actual range attached — is what a decision-maker
can use, and it is not something a single calibrated model can give you.

---

## 6. Where did the data actually teach us anything?

Because every realization carries its own complete K field, we can map the
ensemble spatially.

- **mean** — the calibrated K field, averaged over the ensemble.
- **standard deviation** — how much the realizations still disagree, cell by cell.
- **uncertainty reduction** — the interesting one. Near 1 means the data pinned
  K down there; near 0 means we learned nothing and the prior spread survived
  untouched.

That last map is effectively a report card on the monitoring network: it shows
exactly where the wells are informative and where the model is still guessing.

In [ ]:
LAYER = 0
ies.plot_field('k', stat='mean', which='posterior', layer=LAYER)

In [ ]:
ies.plot_field('k', stat='std', which='posterior', layer=LAYER)

In [ ]:
ies.plot_field('k', stat='reduction', layer=LAYER)

---

## 7. Where is the model still wrong?

Misfit is not uniform in space. This map puts each observation's average residual
(simulated − measured) on the grid: **red is under-simulated, blue is
over-simulated, white is on the money.**

Clusters of one colour are the useful signal — they usually mean something
structural is missing (a boundary in the wrong place, a layer that should be
split) rather than a parameter that needs another nudge.

In [ ]:
ies.plot_obs_residuals()

---

## 8. What to take away

1. **Calibration has many answers, not one.** A single best-fit model hides that.
2. **Run the prior first.** If the ensemble doesn't bracket the measurements
   before calibration, stop and fix the model — that check costs minutes.
3. **The deliverable is a forecast with a range**, not a "calibrated model".
   The narrowing from prior to posterior is what the data bought you.
4. **A perfect fit is a red flag.** Some residual misfit is honest; zero misfit
   usually means the model has been contorted to chase noise.
5. **The uncertainty-reduction map tells you where to drill next** — it shows
   where your existing data is informative and where it isn't.

Everything above came from one declaration of what to adjust, one line to run the
ensemble, and a review object that reopens the finished run from disk.

In [ ]:
# The whole calibration, as it was declared:
#
#     cal = model.pest('demo_ies', start_datetime='2024-01-01')
#     cal.parameterize('k', style='pilotpoints', pp_space=8, layers=[0, 1],
#                      bounds=(0.05, 20.0), physical=(0.001, 300.0), capture=True)
#     cal.parameterize('recharge', style='constant', bounds=(0.3, 3.0), physical=(0.0, 1e-2))
#     cal.observe(head_targets)
#     cal.forecast(forecast_targets)
#     cal.build('demo_ies.pst')
#     ies = cal.run_ies(reals=30, iterations=3, workers=12)
print(ies.settings)